#### 12. I avsnitt 2.2 “Ett kodexempel från början till slut - Huspriser i Kalifornien” så gås ett komplett kodexempel igenom. På valideringsdatan fick vi RMSE Random Forest Regression: 52277.96578719621. Försök få ett bättre resultat genom att exempelvis justera hyperparametrar eller genomföra variabelselektion.

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression

from sklearn.model_selection import train_test_split, GridSearchCV
import time

## Få tillgång till datan

In [44]:
housing_original = pd.read_csv("housing.csv")
housing_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


In [45]:
housing_original.iloc[:, :6].head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population
0,-122.23,37.88,41.0,880.0,129.0,322.0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0
2,-122.24,37.85,52.0,1467.0,190.0,496.0
3,-122.25,37.85,52.0,1274.0,235.0,558.0
4,-122.25,37.85,52.0,1627.0,280.0,565.0


In [46]:
housing_original.iloc[:, 5:].head()

,population,households,median_income,median_house_value,ocean_proximity
0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,496.0,177.0,7.2574,352100.0,NEAR BAY
3,558.0,219.0,5.6431,341300.0,NEAR BAY
4,565.0,259.0,3.8462,342200.0,NEAR BAY


In [47]:
housing_original['ocean_proximity'].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [48]:
housing = housing_original[housing_original['ocean_proximity'] != 'ISLAND']
housing['ocean_proximity'].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
Name: count, dtype: int64

In [49]:
housing = pd.get_dummies(housing, columns=['ocean_proximity'], dtype=int, prefix='dmy')

In [50]:
train_full, test = train_test_split(housing, test_size=0.2, random_state=40)

train, val = train_test_split(train_full, test_size=0.25, random_state=36)
train.info()

<class 'pandas.DataFrame'>
Index: 12381 entries, 12054 to 14298
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           12381 non-null  float64
 1   latitude            12381 non-null  float64
 2   housing_median_age  12381 non-null  float64
 3   total_rooms         12381 non-null  float64
 4   total_bedrooms      12261 non-null  float64
 5   population          12381 non-null  float64
 6   households          12381 non-null  float64
 7   median_income       12381 non-null  float64
 8   median_house_value  12381 non-null  float64
 9   dmy_<1H OCEAN       12381 non-null  int64  
 10  dmy_INLAND          12381 non-null  int64  
 11  dmy_NEAR BAY        12381 non-null  int64  
 12  dmy_NEAR OCEAN      12381 non-null  int64  
dtypes: float64(9), int64(4)
memory usage: 1.3 MB


In [51]:
train = train.dropna()
train.info()

<class 'pandas.DataFrame'>
Index: 12261 entries, 12054 to 14298
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           12261 non-null  float64
 1   latitude            12261 non-null  float64
 2   housing_median_age  12261 non-null  float64
 3   total_rooms         12261 non-null  float64
 4   total_bedrooms      12261 non-null  float64
 5   population          12261 non-null  float64
 6   households          12261 non-null  float64
 7   median_income       12261 non-null  float64
 8   median_house_value  12261 non-null  float64
 9   dmy_<1H OCEAN       12261 non-null  int64  
 10  dmy_INLAND          12261 non-null  int64  
 11  dmy_NEAR BAY        12261 non-null  int64  
 12  dmy_NEAR OCEAN      12261 non-null  int64  
dtypes: float64(9), int64(4)
memory usage: 1.3 MB


In [52]:
val = val.dropna()
test = test.dropna()

In [53]:
# corr_matrix = train.corr()
# corr_matrix["median_house_value"].sort_values(ascending=False)

In [54]:
# attributes = ["median_house_value", "median_income", "total_bedrooms", "dumy_INLAND"]

In [55]:
X_train_full = train_full.drop(columns=['median_house_value'])
y_train_full = train_full['median_house_value']

In [56]:
X_train, y_train = (
    train.drop(columns=['median_house_value']),
    train['median_house_value']
)

X_val, y_val = (
    val.drop(columns=['median_house_value']),
    val['median_house_value']
)

X_test, y_test = (
    test.drop(columns=['median_house_value']),
    test['median_house_value']
)

print(X_train.shape)
print(y_train.shape)

(12261, 12)
(12261,)


In [57]:
start_time = time.time()

rf = RandomForestRegressor( random_state=42, n_jobs=-1)

hyper_grid = {
    'max_depth': [None, 20, 40, 60], 'n_estimators': [50, 100, 200],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4],
    'max_features': [0.7, 1.0]
    }

grid_search = GridSearchCV(estimator=rf, param_grid=hyper_grid, scoring='neg_root_mean_squared_error', cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

end_time = time.time()

excution_time = end_time - start_time
print(f"GridSearchCV fitting took {excution_time:.4f} seconds.")
print()
print("Best Hyperparameters from GridSearchCV: ", grid_search.best_params_)

GridSearchCV fitting took 681.1955 seconds.

Best Hyperparameters from GridSearchCV:  {'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}


In [58]:
rf_pred_val = grid_search.predict(X_val)
print("RMSE Random forest regression: ", root_mean_squared_error(y_val, rf_pred_val))

RMSE Random forest regression:  49350.431891365515


#### Genom de nya Hyperparameterer kunnde jag förbättra RMSE av modellen RandomForestRegressionen , men den tåg längre tid än de gamla Hyperparametrar i boken. Valet i så fall av vilka Hyperparamtrar som ska används i produktionen beror bland annat på slutmåll. 